In [ ]:
from transformers import pipeline, AutoModel, AutoTokenizer, AutoModelForCausalLM
import torch
from typing import Dict, List

In [ ]:
MODEL_NAME = '../models/Qwen/Qwen2.5-7B-Instruct'

In [ ]:
PARAMS = {
    'system_prompt': "You are an AI assistant who helps solve user issues.",
    "item_format": "- [{score}] {document}",
    "user_prompt": 'Answer the question using the available information from the texts in the list below. Each text has a corresponding real-value score of its relevance to the question in square brackets at the beginning. Scores are ranged from 0.0 (the text is not suitable for generating an answer based on it) to 1.0 (the text is suitable for generating an answer based on it). Use this information. Choose texts with high enough relevance scores. If, based on the specified scores, there are no texts in the list that are relevant enough to generate answer based on them, then generate the following answer: "I do not have an answer to your question". Generate answer only in English. Do not duplicate the question in the answer. Generate only the answer to the specified question. Answer need to be short. Do not generate anything extra.',
    "prompt_format": "{user_p}\n\nAvailable information:\n{cnt_list}\n\nQuestion:\n{q}\n\nAnswer:\n",
    'gen_strat': {'max_new_tokens': 1024},
}

In [ ]:
class CustomAgent:
    def __init__(self, model_path, device='cuda:0'):
        self.device = device
        self.model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16).to(device)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)

    def generate(self, user_prompt: str, assistant_prompt: str = None, system_prompt: str = None, gen_strategy: Dict = None):
        messages = [
            {"role": "system", "content": system_prompt if system_prompt is not None else self.config.system_prompt},
            {"role": "user","content": user_prompt}]

        if assistant_prompt is not None:
            messages.insert(1, {"role": "assistant", "content": assistant_prompt})

        prompt = self.tokenizer.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True
        )

        gen_strategy = self.config.gen_strategy if gen_strategy is None else gen_strategy

        inputs = self.tokenizer(
            prompt, return_tensors='pt',
            padding=False, add_special_tokens=False)
        input_ids = inputs['input_ids'].to(self.device)
        attention_mask = inputs['attention_mask'].to(self.device)
        
        output_sequences = self.model.generate(
            input_ids=input_ids, attention_mask=attention_mask, eos_token_id=[self.tokenizer.eos_token_id], 
            pad_token_id=self.tokenizer.eos_token_id, 
            output_logits=True, output_scores=True, output_hidden_states=True,
            return_dict_in_generate=True, **gen_strategy)

        #print(len(inputs['input_ids'][0]))
        #print(len(output_sequences['sequences'][0]))
        
        generated_text = self.tokenizer.decode(output_sequences['sequences'][0][len(input_ids[0]):], skip_special_tokens=True)

        return generated_text, output_sequences

In [ ]:
agent = CustomAgent(MODEL_NAME)

In [ ]:
answer, metainfo = agent.generate(
    user_prompt='What is wrong with humanity?', 
    system_prompt= PARAMS['system_prompt'], 
    gen_strategy=PARAMS['gen_strat'])

In [ ]:
metainfo.keys()

In [ ]:
print(answer)